In [1]:
import csv
from pathlib import Path
import openpyxl
import pandas as pd
from typing import NamedTuple

# 名前付きタプル
# typing.NamedTupleを継承したクラス構文で定義しているので、各フィールドに型ヒント（float）を付けられます。
class MonthlyFigures(NamedTuple):
    # float型（浮動小数点数）
    sales: float       # 売上高
    labor_cost: float  # 人件費
    net_profit: float  # 純利益

class Department:
    def __init__(self, name):
        self.name = name
        # self.records = {} — インスタンス属性recordsを空の辞書で初期化する処理。実際に実行されるのはここだけです。
        # : dict[str, MonthlyFigures] — 型アノテーション。
        # このrecordsが「キーがstr、値がMonthlyFiguresの辞書」であることを示す注釈で、
        # 実行時には何もチェックされません（IDEの補完やmypyのような型チェッカーが読むための情報です）。
        self.records: dict[str, MonthlyFigures] = {} # キー: "2026.04"

    # 指定した"年月"の"売上高", "人件費", "純利益"をrecordsディクショナリへ格納するメソッド
    def add_month(self, ym: str, sales: float, labor_cost: float, net_profit: float) -> None:
        self.records[ym] = MonthlyFigures(sales, labor_cost, net_profit)


# departmentsディクショナリを作成する。キーは部署名(文字列)、値はDepartmentインスタンス
departments: dict[str, Department] = {}

# 全社の数値を計算する関数
def total(departments: dict[str, Department]) -> Department:
    # 名前が"全社"のDepartmentクラスのインスタンスを作成する。
    company = Department("全社")


    # iterable（反復可能オブジェクト）:for文で回せる対象。リスト、辞書、dict_values、文字列など。
    #「要素を順に取り出せる能力がある」というだけで、それ自体は「今どこまで取り出したか」という状態を持ちません。
    # iterator（反復子）:実際に「次の要素を1個取り出す」という操作__next__()を持ち、かつ「今どこまで進んだか」という状態を内部に持つオブジェクト。
    # イメージで言うと、iterableは「本」、iteratorは「今何ページ目を読んでいるかを覚えているしおり付きの読み手」です。
    # 本（iterable）自体は「今どこを読んでいるか」を覚えていません。何度でも最初から読み始められます。
    # iter(x) — iterableなxから、新しいiteratorを作る。しおりを新しく用意して1ページ目を指した状態、というイメージ。
    # next(it) — そのiteratorを1つ進めて、次の要素を返す。しおりを1ページ進める操作。

    # 「全部署が同じ月をそろえて記録している」という前提のもと、代表で1部署だけ選び、
    # その部署の年月キー一覧を「全部署共通の月リスト」としてmonthsに格納しています。
    # この後のコードでfor ym in months:として、全部署の同じ年月のデータをymごとに合算していく際のループ対象として使われます。
    # monthsはdict_keys型。ディクショナリのキー部分だけを覗く窓のようなもの。値にはアクセスできない。
    # months: "2026.04", "2026.05", "2026.06", "2026.07", "2026.08"
    months = next(iter(departments.values())).records.keys()

    for ym in months:
        # departments.values() — 全部署のDepartmentインスタンス（dict_values）(=departmentsディクショナリの値全部を指す)
        # for d in departments.values() — それを1部署ずつdとして取り出す
        # d.records[ym] — その部署の、今見ている年月ymのMonthlyFigures
        # d.records[ym].sales — そのMonthlyFiguresの売上高
        # sum(...) — 全部署分の.salesを合計する
        sales_sum = sum(d.records[ym].sales for d in departments.values())
        labor_sum = sum(d.records[ym].labor_cost for d in departments.values())
        profit_sum = sum(d.records[ym].net_profit for d in departments.values())
        # Departmentクラスのadd_monthメソッドを呼び出し
        company.add_month(ym, sales_sum, labor_sum, profit_sum)

    return company



# 例: 2026.04.csv, 2026.05.csv, ... の順
# Path.cwd().glob("*.csv")でカレントディレクトリの.csvファイルを全部拾い、
# sorted()でファイル名の文字コード順（アルファベット順）に並べ替えています。
# sorted()はlist型を返す→csv_filesはlist型で、中身の要素はPathオブジェクト
csv_files = sorted(Path.cwd().glob("*.csv"))

for csv_path in csv_files:
    # withは「開いて→使って→（エラーが起きても必ず）閉じる」
    # withを抜けるときf.close()する。
    with open(csv_path, newline="", encoding="cp932") as f:
        # csv.DictReaderというクラスのインスタンスを作り、
        # readerという変数に代入しています。
        # 引数として渡したf（開いたファイルオブジェクト）
        # から1行ずつ読み込み、1行目を見出し（ヘッダー）として自動的に扱い、
        # 以降の各行を「見出し名をキーにしたdict」に変換して返してくれるオブジェクトです。
        reader = csv.DictReader(f)
        # reader（DictReader）から1行ずつ取り出す。
        # rowにはその行のdictが入る。
        # 例えば{'年月': '2026.04', '部署': '技術部', '売上高': '450', '人件費': '380', '純利益': '70'}のような形。
        for row in reader:
            # その行の「部署」列の値（例: '技術部'）を取り出してnameに入れる。
            # これは後でdepartments辞書のキーとして使う。
            name = row["部署"]
            # departmentsディクショナリにまだその部署名のキーが存在しない場合だけ、
            # 新しくDepartmentインスタンスを作って登録する。
            if name not in departments:
                departments[name] = Department(name)
            # CSVの1行分のデータを、対応するDepartmentインスタンスのrecordsに登録している処理です。
            # 1つ前でname = row["部署"]として部署名を取り出し、departments[name]がまだなければ新規作成しています。
            departments[name].add_month(
                row["年月"],
                float(row["売上高"]),
                float(row["人件費"]),
                float(row["純利益"]),
            )

# 全社の数値を計算する
company = total(departments)

# 収支報告書.xlsxを作成する。
def write_department_sheet(wb, department: Department) -> None:
    ws = wb.create_sheet(title=department.name)
    # ws.append(...)は、シートの一番下（この時点ではまだ何もないので1行目）に、
    # リストの中身を1行分のセルとして追加するメソッドです。リストの各要素が左から順にA列、B列、C列、D列に入ります。
    # つまりこの2行で、「部署名のシートを新規作成し、1行目に見出し行（部署名＋列名）を書き込む」という処理をしています。
    ws.append([department.name, "売上高", "人件費", "純利益"])

    # enumerate()は、iterableの各要素に「連番（インデックス）」を自動で付けながら列挙する、Pythonの組み込み関数です。
    # 戻り値はenumerateオブジェクト（これ自体がイテレータ）で、中身は(番号, 要素)というタプルの並びになります。
    # department.records.items()について、recordsはdict[str, MonthlyFigures]（例: {"2026.04": MonthlyFigures(...), "2026.05": MonthlyFigures(...)}）なので、
    # .items()は「キーと値のペア」を返すビュー（dict_items）です。1つの要素は("2026.04", MonthlyFigures(...))のようなタプルになります。
    # start=2で2行目から開始
    for row_num, (ym, fig) in enumerate(department.records.items(), start=2):
        ws.append([float(ym), fig.sales, fig.labor_cost, f"=B{row_num}-C{row_num}"])

    last_row = row_num
    ws.append(["小計", f"=SUM(B2:B{last_row})", f"=SUM(C2:C{last_row})", f"=SUM(D2:D{last_row})"])


wb = openpyxl.Workbook()
# 新規Workbookに自動で付いてくる空シートを削除
del wb["Sheet"]

# "全社"はdepartmentsディクショナリに含まれていないためここで書き込む
# "全社"についてエクセルのシートに書き込む
write_department_sheet(wb, company)

# 96,97行目でdepartmentsディクショナリに"技術部"～"営業部"までの5つの部署のインスタンスを作っている。
# "技術部"～"営業部"までの5つの部署についてエクセルの各シートに書き込む
for department in departments.values():
    write_department_sheet(wb, department)

wb.save("収支報告書.xlsx")